# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

# API Project - Bus Bunching
### Multi-agency Comparsion (WMATA, MBTA, MTA)
Question: What route and temporal factors are associated with bus bunching on high-frequency corridors, and do these relationships vary across different transit agencies (NYC, Boston, and DC)?

In [45]:
import pandas as pd
import numpy as np
import requests
import ratelim
import tenacity

### 1) WMATA API
#### Bus Routes: D40, D60, D80, C53, C61, D20

In [46]:
WMATA_KEY = "5ec08fcd21744590b1a0f655710c4c91"

In [47]:
wmata_url = "https://api.wmata.com/Bus.svc/json/jBusPositions"
wmata_params = {"RouteID": "D40"}  
wmata_headers = {"api_key": WMATA_KEY}

requests.get(wmata_url, params=wmata_params, headers=wmata_headers)

<Response [200]>

In [48]:
wmata_response = requests.get(wmata_url, params=wmata_params, headers=wmata_headers)
wmata_response.json()

{'BusPositions': [{'VehicleID': '5536',
   'Lat': 38.89257049560547,
   'Lon': -77.02388763427734,
   'Deviation': 8.0,
   'DateTime': '2026-03-31T20:48:08',
   'TripID': '28868020',
   'RouteID': 'D40',
   'DirectionNum': 1,
   'DirectionText': 'SOUTH',
   'TripHeadsign': 'ARCHIVES',
   'TripStartTime': '2026-03-31T19:48:00',
   'TripEndTime': '2026-03-31T20:37:00',
   'BlockNumber': 'M615'},
  {'VehicleID': '5517',
   'Lat': 38.90422075445002,
   'Lon': -77.02191162109375,
   'Deviation': 7.0,
   'DateTime': '2026-03-31T20:48:09',
   'TripID': '32430020',
   'RouteID': 'D40',
   'DirectionNum': 1,
   'DirectionText': 'SOUTH',
   'TripHeadsign': 'ARCHIVES',
   'TripStartTime': '2026-03-31T20:00:00',
   'TripEndTime': '2026-03-31T20:49:00',
   'BlockNumber': 'M619'},
  {'VehicleID': '5513',
   'Lat': 38.96554347446987,
   'Lon': -77.02764769962856,
   'Deviation': 11.0,
   'DateTime': '2026-03-31T20:47:57',
   'TripID': '163020',
   'RouteID': 'D40',
   'DirectionNum': 0,
   'Direction

#### Vars Used: 
- `Deviation` — the primary outcome, minutes late/early so no GTFS needed
- `DateTime` — timestamp will be used for peak hour flag
- `RouteID`, `DirectionNum`, `VehicleID`, `TripID` — route/trip identifiers (`DirectionNum`, 0=North 1=South)
- `TripStartTime` — scheduled headway, will be used to calc the gap between consecutive trips
- `Lat`, `Lon` — Vehicle position

In [49]:
wmata_fields = {
    'VehicleID': 'vehicle_id',
    'Lat': 'lat',
    'Lon': 'lon',
    'Deviation': 'deviation',
    'DateTime': 'timestamp',
    'TripID': 'trip_id',
    'RouteID': 'route_id',
    'DirectionNum': 'direction',
    'TripStartTime': 'trip_start_time'
}

wmata_positions = wmata_response.json()['BusPositions']

df = pd.DataFrame(wmata_positions)[list(wmata_fields.keys())].rename(columns=wmata_fields)
df['agency'] = 'WMATA'

print(df)
print(df.dtypes) #Checks for var types (timestamp and trip_start_time)

  vehicle_id        lat        lon  deviation            timestamp   trip_id  \
0       5536  38.892570 -77.023888        8.0  2026-03-31T20:48:08  28868020   
1       5517  38.904221 -77.021912        7.0  2026-03-31T20:48:09  32430020   
2       5513  38.965543 -77.027648       11.0  2026-03-31T20:47:57    163020   
3       7258  38.915917 -77.021919        1.0  2026-03-31T20:47:53    608020   
4       5521  38.966916 -77.027539       -2.0  2026-03-31T20:48:15   5155020   
5       5509  38.928438 -77.023146        2.0  2026-03-31T20:47:48  20886020   
6       5520  38.981871 -77.026543        3.0  2026-03-31T20:48:15   7858020   
7       5512  38.893245 -77.023895       24.0  2026-03-31T20:48:14  34445020   
8       5533  38.993618 -77.029640        0.0  2026-03-31T20:48:00  13320020   

  route_id  direction      trip_start_time agency  
0      D40          1  2026-03-31T19:48:00  WMATA  
1      D40          1  2026-03-31T20:00:00  WMATA  
2      D40          0  2026-03-31T20:00:00 

In [50]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
print(df.dtypes)

vehicle_id                 object
lat                       float64
lon                       float64
deviation                 float64
timestamp          datetime64[ns]
trip_id                    object
route_id                   object
direction                   int64
trip_start_time    datetime64[ns]
agency                     object
dtype: object


In [51]:
routes = ['D40', 'D60', 'D80', 'C53', 'C61', 'D20']
dfs = []

for route in routes:
    wmata_response = requests.get(wmata_url, params={"RouteID": route}, headers=wmata_headers)
    wmata_positions = wmata_response.json()['BusPositions']
    df = pd.DataFrame(wmata_positions)[list(wmata_fields.keys())].rename(columns=wmata_fields)
    df['agency'] = 'WMATA'
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
    dfs.append(df)

wmata_df = pd.concat(dfs, ignore_index=True)

In [52]:
wmata_df

,vehicle_id,lat,lon,deviation,timestamp,trip_id,route_id,direction,trip_start_time,agency
0,5536,38.892570,-77.023888,8.0,2026-03-31 20:48:08,28868020,D40,1,2026-03-31 19:48:00,WMATA
1,5517,38.904221,-77.021912,7.0,2026-03-31 20:48:09,32430020,D40,1,2026-03-31 20:00:00,WMATA
2,5513,38.965543,-77.027648,11.0,2026-03-31 20:47:57,163020,D40,0,2026-03-31 20:00:00,WMATA
3,7258,38.915917,-77.021919,1.0,2026-03-31 20:47:53,608020,D40,1,2026-03-31 20:12:00,WMATA
4,5521,38.966916,-77.027539,-2.0,2026-03-31 20:48:15,5155020,D40,0,2026-03-31 20:12:00,WMATA
5,5509,38.928438,-77.023146,2.0,2026-03-31 20:47:48,20886020,D40,0,2026-03-31 20:24:00,WMATA
6,5520,38.981871,-77.026543,3.0,2026-03-31 20:48:15,7858020,D40,1,2026-03-31 20:36:00,WMATA
7,5512,38.893245,-77.023895,24.0,2026-03-31 20:48:14,34445020,D40,0,2026-03-31 20:36:00,WMATA
8,5533,38.993618,-77.029640,0.0,2026-03-31 20:48:00,13320020,D40,1,2026-03-31 20:48:00,WMATA
9,5515,38.902475,-77.030637,-5.0,2026-03-31 20:47:19,15851020,D60,1,2026-03-31 20:00:00,WMATA


In [53]:
print(wmata_df['route_id'].unique()) #Checking that all the routes i need are in the df

['D40' 'D60' 'D80' 'C53' 'C61' 'D20']


### 2) MBTA API
#### Bus Routes: Route 1, 23, 28, 39, 57, 66

In [54]:
MBTA_KEY = '4aa18b9edd1341e383dc8fe55a788cca'

In [55]:
#Test
mbta_url = "https://api-v3.mbta.com/vehicles"
mbta_params = {"filter[route]": "1"}
mbta_headers = {"x-api-key": MBTA_KEY}

requests.get(mbta_url, params=mbta_params, headers=mbta_headers)

<Response [200]>

In [56]:
mbta_response = requests.get(mbta_url, params=mbta_params, headers=mbta_headers)
mbta_response.json()

{'data': [{'attributes': {'bearing': 90,
    'carriages': [],
    'current_status': 'STOPPED_AT',
    'current_stop_sequence': 1,
    'direction_id': 0,
    'label': '1922',
    'latitude': 42.32983,
    'longitude': -71.08371,
    'occupancy_status': 'MANY_SEATS_AVAILABLE',
    'revenue': 'REVENUE',
    'speed': None,
    'updated_at': '2026-03-31T20:48:04-04:00'},
   'id': 'y1922',
   'links': {'self': '/vehicles/y1922'},
   'relationships': {'route': {'data': {'id': '1', 'type': 'route'}},
    'stop': {'data': {'id': '64', 'type': 'stop'}},
    'trip': {'data': {'id': '73601905', 'type': 'trip'}}},
   'type': 'vehicle'},
  {'attributes': {'bearing': 332,
    'carriages': [],
    'current_status': 'IN_TRANSIT_TO',
    'current_stop_sequence': 8,
    'direction_id': 0,
    'label': '1875',
    'latitude': 42.336768,
    'longitude': -71.077224,
    'occupancy_status': 'MANY_SEATS_AVAILABLE',
    'revenue': 'REVENUE',
    'speed': None,
    'updated_at': '2026-03-31T20:48:23-04:00'},
 

#### Vars Used
- `id` — vehicle_id
- `attributes.latitude`, `attributes.longitude` — lat, lon
- `attributes.direction_id` — direction (already 0/1, but doesn't tell me which direction that is)
- `attributes.updated_at` — timestamp
- `attributes.current_stop_sequence` — stop_sequence (not available in WMATA)
- `relationships.trip.data.id` — trip_id
- `relationships.route.data.id` — route_id

#### Notes
- No deviation field — must be computed from GTFS static feed in R
- I'll also need GTFS static for the direction north or south bound
- This nested unlike WMATA which was more straight forward

In [57]:
mbta_positions = mbta_response.json()['data']
mbta_positions

mbta_fields = []
for vehicle in mbta_positions:
    row = {
        'vehicle_id': vehicle['id'],
        'lat': vehicle['attributes']['latitude'],
        'lon': vehicle['attributes']['longitude'],
        'direction': vehicle['attributes']['direction_id'],
        'timestamp': vehicle['attributes']['updated_at'],
        'stop_sequence': vehicle['attributes']['current_stop_sequence'],
        'trip_id': vehicle['relationships']['trip']['data']['id'],
        'route_id': vehicle['relationships']['route']['data']['id']
    }
    mbta_fields.append(row)

mbta_df = pd.DataFrame(mbta_fields)
mbta_df['agency'] = 'MBTA'
mbta_df['timestamp'] = pd.to_datetime(mbta_df['timestamp'])

mbta_df

,vehicle_id,lat,lon,direction,timestamp,stop_sequence,trip_id,route_id,agency
0,y1922,42.329830,-71.083710,0,2026-03-31 20:48:04-04:00,1,73601905,1,MBTA
1,y1875,42.336768,-71.077224,0,2026-03-31 20:48:23-04:00,8,73601895,1,MBTA
2,y1864,42.357648,-71.092876,1,2026-03-31 20:48:20-04:00,11,73602195,1,MBTA
3,y1849,42.340768,-71.081972,1,2026-03-31 20:48:16-04:00,16,73602191,1,MBTA
4,y1846,42.362598,-71.099434,0,2026-03-31 20:48:24-04:00,17,73601894,1,MBTA
5,y1729,42.373356,-71.117745,1,2026-03-31 20:48:18-04:00,1,73602194,1,MBTA


In [58]:
mbta_df.dtypes
mbta_df['timestamp'] = mbta_df['timestamp'].dt.tz_localize(None) # Making time zone consistent
mbta_df.dtypes

vehicle_id               object
lat                     float64
lon                     float64
direction                 int64
timestamp        datetime64[ns]
stop_sequence             int64
trip_id                  object
route_id                 object
agency                   object
dtype: object

In [59]:
mbta_routes = ['1', '23', '28', '39', '57', '66']
mbta_dfs = []

for route in mbta_routes:
    mbta_response = requests.get(mbta_url, params={"filter[route]": route}, headers=mbta_headers)
    mbta_fields = []
    for vehicle in mbta_response.json()['data']:
        row = {
            'vehicle_id': vehicle['id'],
            'lat': vehicle['attributes']['latitude'],
            'lon': vehicle['attributes']['longitude'],
            'direction': vehicle['attributes']['direction_id'],
            'timestamp': vehicle['attributes']['updated_at'],
            'stop_sequence': vehicle['attributes']['current_stop_sequence'],
            'trip_id': vehicle['relationships']['trip']['data']['id'],
            'route_id': vehicle['relationships']['route']['data']['id']
        }
        mbta_fields.append(row)
    mbta_df = pd.DataFrame(mbta_fields)
    mbta_df['agency'] = 'MBTA'
    mbta_df['timestamp'] = pd.to_datetime(mbta_df['timestamp']).dt.tz_localize(None)
    mbta_dfs.append(mbta_df)

mbta_df = pd.concat(mbta_dfs, ignore_index=True)
mbta_df

,vehicle_id,lat,lon,direction,timestamp,stop_sequence,trip_id,route_id,agency
0,y1922,42.329830,-71.083710,0,2026-03-31 20:48:04,1,73601905,1,MBTA
1,y1875,42.336768,-71.077224,0,2026-03-31 20:48:23,8,73601895,1,MBTA
2,y1864,42.357648,-71.092876,1,2026-03-31 20:48:20,11,73602195,1,MBTA
3,y1849,42.340768,-71.081972,1,2026-03-31 20:48:16,16,73602191,1,MBTA
4,y1846,42.362598,-71.099434,0,2026-03-31 20:48:24,17,73601894,1,MBTA
5,y1729,42.373356,-71.117745,1,2026-03-31 20:48:18,1,73602194,1,MBTA
6,y1918,42.283650,-71.063910,1,2026-03-31 20:48:12,1,73601420,23,MBTA
7,y1910,42.308510,-71.083170,0,2026-03-31 20:48:08,16,73603918,23,MBTA
8,y1862,42.325546,-71.083475,0,2026-03-31 20:48:18,7,73603917,23,MBTA
9,y1833,42.331588,-71.094135,1,2026-03-31 20:48:23,29,73601418,23,MBTA


### 3) MTA API
#### Bus Routes: B44, B46, B41, Bx12, M14A, M104A

In [60]:
MTA_KEY = '7027c575-3c63-477c-91d8-3d404472e8a0'

In [61]:
mta_url = "http://bustime.mta.info/api/siri/vehicle-monitoring.json"
mta_params = {
    "key": MTA_KEY,
    "LineRef": "MTA NYCT_B44"
}
requests.get(mta_url, params=mta_params)

<Response [200]>

In [62]:
mta_response = requests.get(mta_url, params=mta_params)
mta_response.json()

{'Siri': {'ServiceDelivery': {'ResponseTimestamp': '2026-03-31T20:48:28.930-04:00',
   'VehicleMonitoringDelivery': [{'VehicleActivity': [{'MonitoredVehicleJourney': {'LineRef': 'MTA NYCT_B44',
        'DirectionRef': '1',
        'FramedVehicleJourneyRef': {'DataFrameRef': '2026-03-31',
         'DatedVehicleJourneyRef': 'MTA NYCT_FB_A6-Weekday-SDon-123700_B44_335'},
        'JourneyPatternRef': 'MTA_B44O0126',
        'PublishedLineName': 'B44',
        'OperatorRef': 'MTA NYCT',
        'OriginRef': 'MTA_308704',
        'DestinationName': 'SHEEPSHEAD BAY KNAPP ST via NOSTRAND',
        'OriginAimedDepartureTime': '2026-03-31T20:37:00.000-04:00',
        'SituationRef': [{'SituationSimpleRef': 'MTA NYCT_lmm:planned_work:22527'},
         {'SituationSimpleRef': 'MTA NYCT_lmm:planned_work:15159'}],
        'Monitored': True,
        'VehicleLocation': {'Longitude': -73.953665, 'Latitude': 40.699878},
        'Bearing': 305.83765,
        'ProgressRate': 'noProgress',
        'Progress

#### Fields Used
- `VehicleRef` →  `"MTA NYCT_"` prefix
- `VehicleLocation.Latitude`, `VehicleLocation.Longitude` 
- `DirectionRef` → direction (comes as string 0/1)
- `RecordedAtTime` → timestamp
- `LineRef` → route_id (strip `"MTA NYCT_"` prefix)
- `MonitoredCall.Extensions.Distances.CallDistanceAlongRoute` → stop_sequence
- `MonitoredCall.ExpectedArrivalTime` - `MonitoredCall.AimedArrivalTime` → used to calc deviation

In [63]:
mta_vehicles = mta_response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]['VehicleActivity']
mta_vehicles

[{'MonitoredVehicleJourney': {'LineRef': 'MTA NYCT_B44',
   'DirectionRef': '1',
   'FramedVehicleJourneyRef': {'DataFrameRef': '2026-03-31',
    'DatedVehicleJourneyRef': 'MTA NYCT_FB_A6-Weekday-SDon-123700_B44_335'},
   'JourneyPatternRef': 'MTA_B44O0126',
   'PublishedLineName': 'B44',
   'OperatorRef': 'MTA NYCT',
   'OriginRef': 'MTA_308704',
   'DestinationName': 'SHEEPSHEAD BAY KNAPP ST via NOSTRAND',
   'OriginAimedDepartureTime': '2026-03-31T20:37:00.000-04:00',
   'SituationRef': [{'SituationSimpleRef': 'MTA NYCT_lmm:planned_work:22527'},
    {'SituationSimpleRef': 'MTA NYCT_lmm:planned_work:15159'}],
   'Monitored': True,
   'VehicleLocation': {'Longitude': -73.953665, 'Latitude': 40.699878},
   'Bearing': 305.83765,
   'ProgressRate': 'noProgress',
   'ProgressStatus': 'layover',
   'BlockRef': 'MTA NYCT_FB_A6-Weekday-SDon_E_FB_34080_B44-311',
   'VehicleRef': 'MTA NYCT_7957',
   'MonitoredCall': {'AimedArrivalTime': '2026-03-31T20:37:00.000-04:00',
    'ExpectedArrivalTime

In [64]:
mta_routes = ['B44', 'B46', 'B41', 'Bx12', 'M14A', 'M104']
mta_dfs = []

for route in mta_routes:
    mta_params = {"key": MTA_KEY, "LineRef": f"MTA NYCT_{route}"} #This needs to be in the loop not outside
    response = requests.get(mta_url, params=mta_params)
    delivery = response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]
    mta_vehicles = delivery.get('VehicleActivity', [])  # creating an empty list if no active vehicles (this caused loop to not work)
    
    mta_rows = []
    for vehicle in mta_vehicles:
        mvj = vehicle['MonitoredVehicleJourney']  # the path to data
        if 'MonitoredCall' not in mvj:  # skip buses in layover (this also caused loop to not work)
            continue
        mc = mvj['MonitoredCall']  # the path to data

        if 'ExpectedArrivalTime' not in mc:  # skip vehicles with no prediction (Dropping, won't be able to calc dev)
            continue
        
         # parse both times so it can be subtracted to get dev
        aimed = pd.to_datetime(mc['AimedArrivalTime'])
        expected = pd.to_datetime(mc['ExpectedArrivalTime'])
        
        row = {
            'vehicle_id': mvj['VehicleRef'].replace('MTA NYCT_', ''),  # removing prefix
            'lat': mvj['VehicleLocation']['Latitude'],
            'lon': mvj['VehicleLocation']['Longitude'],
            'direction': int(mvj['DirectionRef']),  # making the string "0"/"1" a int
            'timestamp': pd.to_datetime(vehicle['RecordedAtTime']),
            'route_id': mvj['LineRef'].replace('MTA NYCT_', ''),  # removing prefix
            'call_distance': mc['Extensions']['Distances']['CallDistanceAlongRoute'],  # stop_sequence equivalent
            'deviation_min': (expected - aimed).total_seconds() / 60  # positive = late, negative = early
        }
        mta_rows.append(row)
    
    df = pd.DataFrame(mta_rows)
    if df.empty:  # skip if no valid vehicles for this route
        continue
    df['agency'] = 'MTA'
    df['timestamp'] = df['timestamp'].dt.tz_localize(None)  # making timezone to match WMATA/MBTA
    mta_dfs.append(df)

mta_df = pd.concat(mta_dfs, ignore_index=True)
mta_df

,vehicle_id,lat,lon,direction,timestamp,route_id,call_distance,deviation_min,agency
0,7957,40.699878,-73.953665,1,2026-03-31 20:48:21.000,B44,0.00,11.502633,MTA
1,7314,40.645319,-73.948979,1,2026-03-31 20:48:06.000,B44,6166.82,2.069300,MTA
2,7330,40.634265,-73.947797,0,2026-03-31 20:48:11.000,B44,6376.46,10.035967,MTA
3,7949,40.641163,-73.948523,1,2026-03-31 20:48:27.474,B44,6630.35,6.738267,MTA
4,7015,40.585935,-73.939123,1,2026-03-31 20:48:12.000,B44,12853.20,6.485967,MTA
5,7617,40.690152,-73.955267,0,2026-03-31 20:48:04.000,B44,10849.80,0.040433,MTA
6,7316,40.613248,-73.944264,0,2026-03-31 20:48:05.000,B44,4147.12,-0.494817,MTA
7,7327,40.680533,-73.952325,0,2026-03-31 20:48:26.747,B44,12329.21,-1.966633,MTA
8,7635,40.653389,-73.946937,0,2026-03-31 20:48:13.000,B44,6235.40,6.932133,MTA
9,7615,40.668242,-73.950630,1,2026-03-31 20:48:13.000,B44,3705.61,9.485800,MTA


In [65]:
#Checking routes
print(mta_df.groupby('route_id').size())

route_id
B41     20
B44     13
B46     15
M104     7
dtype: int64


In [66]:
# Bx12 and M14 missing

requests.get(mta_url, params={"key": MTA_KEY, "LineRef": "MTA NYCT_Bx12"})
requests.get(mta_url, params={"key": MTA_KEY, "LineRef": "MTA NYCT_M14A"})

<Response [200]>

In [67]:
bx12_response = requests.get(mta_url, params={"key": MTA_KEY, "LineRef": "MTA NYCT_Bx12"})
print(bx12_response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0])

{'ResponseTimestamp': '2026-03-31T20:48:30.324-04:00', 'ErrorCondition': {'OtherError': {'ErrorText': 'No such route: MTA NYCT_Bx12.'}, 'Description': 'No such route: MTA NYCT_Bx12.'}}


In [68]:
m14a_response = requests.get(mta_url, params={"key": MTA_KEY, "LineRef": "MTA NYCT_M14A"})
print(m14a_response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0])

{'ResponseTimestamp': '2026-03-31T20:48:30.381-04:00', 'ErrorCondition': {'OtherError': {'ErrorText': 'No such route: MTA NYCT_M14A.'}, 'Description': 'No such route: MTA NYCT_M14A.'}}


In [69]:
#Testing for correct route names
test_response = requests.get(
    "http://bustime.mta.info/api/where/routes-for-agency/MTA%20NYCT.json",
    params={"key": MTA_KEY}
)
routes = test_response.json()['data']['list']
for r in routes:
    if 'x12' in r['id'].lower() or 'm14' in r['id'].lower():
        print(r['id'], r['shortName'])

MTA NYCT_BX12+ Bx12-SBS
MTA NYCT_BX12 Bx12
MTA NYCT_M14A+ M14A-SBS
MTA NYCT_M14D+ M14D-SBS


In [70]:
mta_routes = ['B44', 'B46', 'B41', 'BX12', 'M14A+', 'M104']
mta_dfs = []

for route in mta_routes:
    mta_params = {"key": MTA_KEY, "LineRef": f"MTA NYCT_{route}"} #This needs to be in the loop not outside
    response = requests.get(mta_url, params=mta_params)
    delivery = response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]
    mta_vehicles = delivery.get('VehicleActivity', [])  # creating an empty list if no active vehicles (this caused loop to not work)
    
    mta_rows = []
    for vehicle in mta_vehicles:
        mvj = vehicle['MonitoredVehicleJourney']  # the path to data
        if 'MonitoredCall' not in mvj:  # skip buses in layover (this also caused loop to not work)
            continue
        mc = mvj['MonitoredCall']  # the path to data

        if 'ExpectedArrivalTime' not in mc:  # skip vehicles with no prediction (Dropping, won't be able to calc dev)
            continue
        
         # parse both times so it can be subtracted to get dev
        aimed = pd.to_datetime(mc['AimedArrivalTime'])
        expected = pd.to_datetime(mc['ExpectedArrivalTime'])
        
        row = {
            'vehicle_id': mvj['VehicleRef'].replace('MTA NYCT_', ''),  # removing prefix
            'lat': mvj['VehicleLocation']['Latitude'],
            'lon': mvj['VehicleLocation']['Longitude'],
            'direction': int(mvj['DirectionRef']),  # making the string "0"/"1" a int
            'timestamp': pd.to_datetime(vehicle['RecordedAtTime']),
            'route_id': mvj['LineRef'].replace('MTA NYCT_', ''),  # removing prefix
            'call_distance': mc['Extensions']['Distances']['CallDistanceAlongRoute'],  # stop_sequence equivalent
            'deviation': (expected - aimed).total_seconds() / 60  # positive = late, negative = early (var created)
        }
        mta_rows.append(row)
    
    df = pd.DataFrame(mta_rows)
    if df.empty:  # skip if no valid vehicles for this route
        continue
    df['agency'] = 'MTA'
    df['timestamp'] = df['timestamp'].dt.tz_localize(None)  # making timezone to match WMATA/MBTA
    mta_dfs.append(df)

mta_df = pd.concat(mta_dfs, ignore_index=True)
mta_df

,vehicle_id,lat,lon,direction,timestamp,route_id,call_distance,deviation,agency
0,7957,40.699878,-73.953665,1,2026-03-31 20:48:21.000,B44,0.00,11.525567,MTA
1,7330,40.634265,-73.947797,0,2026-03-31 20:48:11.000,B44,6376.46,10.058900,MTA
2,7949,40.641163,-73.948523,1,2026-03-31 20:48:27.474,B44,6630.35,6.738267,MTA
3,7314,40.645319,-73.948979,1,2026-03-31 20:48:06.000,B44,6166.82,2.092233,MTA
4,7015,40.585935,-73.939123,1,2026-03-31 20:48:12.000,B44,12853.20,6.508900,MTA
...,...,...,...,...,...,...,...,...,...
65,9769,40.803693,-73.967169,1,2026-03-31 20:48:11.115,M104,2561.91,2.326350,MTA
66,9767,40.799582,-73.968180,0,2026-03-31 20:48:13.776,M104,5488.53,3.018917,MTA
67,9773,40.779709,-73.981780,1,2026-03-31 20:48:08.718,M104,5575.55,-1.247567,MTA
68,9446,40.765377,-73.980143,1,2026-03-31 20:48:27.364,M104,7520.47,2.806467,MTA


In [71]:
print(mta_df.groupby('route_id').size())

route_id
B41      20
B44      13
B46      15
BX12      8
M104      7
M14A+     7
dtype: int64


In [72]:
# Checking that all dfs have the came columns

print("WMATA columns:", wmata_df.columns.tolist())
print("MBTA columns:", mbta_df.columns.tolist())
print("MTA columns:", mta_df.columns.tolist())

WMATA columns: ['vehicle_id', 'lat', 'lon', 'deviation', 'timestamp', 'trip_id', 'route_id', 'direction', 'trip_start_time', 'agency']
MBTA columns: ['vehicle_id', 'lat', 'lon', 'direction', 'timestamp', 'stop_sequence', 'trip_id', 'route_id', 'agency']
MTA columns: ['vehicle_id', 'lat', 'lon', 'direction', 'timestamp', 'route_id', 'call_distance', 'deviation', 'agency']


In [73]:
#Checking var types
print("WMATA\n", wmata_df.dtypes)
print("\nMBTA\n", mbta_df.dtypes)
print("\nMTA\n", mta_df.dtypes)

WMATA
 vehicle_id                 object
lat                       float64
lon                       float64
deviation                 float64
timestamp          datetime64[ns]
trip_id                    object
route_id                   object
direction                   int64
trip_start_time    datetime64[ns]
agency                     object
dtype: object

MBTA
 vehicle_id               object
lat                     float64
lon                     float64
direction                 int64
timestamp        datetime64[ns]
stop_sequence             int64
trip_id                  object
route_id                 object
agency                   object
dtype: object

MTA
 vehicle_id               object
lat                     float64
lon                     float64
direction                 int64
timestamp        datetime64[ns]
route_id                 object
call_distance           float64
deviation               float64
agency                   object
dtype: object


# API Pull - Continuous Loop

WMATA
- High Frequency Routes: D40, D60, D80, C53, C61, D20, P40
- Low Frequency Routes: C81, P16, D24, P14, C23, P20, C35

MBTA
- High Frequency Routes: 1, 23, 28, 39, 57, 66, 111
- Low Frequency Routes: 64, 201, 50, 42, 87, 85, 70

MTA
- High Frequency Routes: B44, B46, B41, BX12, M14A+, M23+, Q98
- Low Frequency Routes: B32, M21, BX42, M50, Q67, B84, Q42

 Note: I added low frequency routes

In [74]:
#Updated Routes
wmata_routes = ['D40', 'D60', 'D80', 'C53', 'C61', 'D20', 'P40', 'C81', 'P16', 'D24', 'P14', 'C23', 'P20', 'C35']
mbta_routes = ['1', '23', '28', '39', '57', '66', '111', '64', '201', '50', '42', '87', '85', '70']
mta_routes = ['B44', 'B46', 'B41', 'BX12', 'M14A+', 'M23+', 'Q98', 'B32', 'M21', 'BX42', 'M50', 'Q67', 'B84', 'Q42']

In [75]:
wmata_fields = {
    'VehicleID': 'vehicle_id',
    'Lat': 'lat',
    'Lon': 'lon',
    'Deviation': 'deviation',
    'DateTime': 'timestamp',
    'TripID': 'trip_id',
    'RouteID': 'route_id',
    'DirectionNum': 'direction',
    'TripStartTime': 'trip_start_time'
}

In [81]:
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)  # creating data folder to store csv

wmata_file = os.path.join(output_dir, "wmata.csv")
mbta_file = os.path.join(output_dir, "mbta.csv")
mta_file = os.path.join(output_dir, "mta.csv")

while True:
    try:
# WMATA
        try: #this allows for the loop to contuine if the pull fails for one agency
            wmata_dfs = []
            for route in wmata_routes:
                wmata_response = requests.get(wmata_url, params={"RouteID": route}, headers=wmata_headers)
                wmata_positions = wmata_response.json()['BusPositions']
                if not wmata_positions:  # skip if no active vehicles
                    continue
                df = pd.DataFrame(wmata_positions)[list(wmata_fields.keys())].rename(columns=wmata_fields)
                df['agency'] = 'WMATA'
                df['timestamp'] = pd.to_datetime(df['timestamp'])
                df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
                wmata_dfs.append(df)
            if wmata_dfs:
                wmata_df = pd.concat(wmata_dfs, ignore_index=True)
                wmata_df.to_csv(wmata_file, mode='a', header=not os.path.exists(wmata_file), index=False) # appending pulls to exsiting file
        except Exception as e:
            print(f"WMATA error: {e}") #if an error this will print it and move on

# MBTA
        try:
            mbta_dfs = []
            for route in mbta_routes:
                mbta_response = requests.get(mbta_url, params={"filter[route]": route}, headers=mbta_headers)
                mbta_vehicles = mbta_response.json().get('data', [])  # empty list if no active vehicles
                if not mbta_vehicles:  # skip if no active vehicles
                    continue
                mbta_fields = []
                for vehicle in mbta_vehicles:
                    row = {
                        'vehicle_id': vehicle['id'],
                        'lat': vehicle['attributes']['latitude'],
                        'lon': vehicle['attributes']['longitude'],
                        'direction': vehicle['attributes']['direction_id'],
                        'timestamp': vehicle['attributes']['updated_at'],
                        'stop_sequence': vehicle['attributes']['current_stop_sequence'],
                        'trip_id': vehicle['relationships']['trip']['data']['id'],
                        'route_id': vehicle['relationships']['route']['data']['id']
                    }
                    mbta_fields.append(row)
                df = pd.DataFrame(mbta_fields)
                if df.empty:
                    continue
                df['agency'] = 'MBTA'
                df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
                mbta_dfs.append(df)
            if mbta_dfs:
                mbta_df = pd.concat(mbta_dfs, ignore_index=True)
                mbta_df.to_csv(mbta_file, mode='a', header=not os.path.exists(mbta_file), index=False)
        except Exception as e:
            print(f"MBTA error: {e}")

# MTA
        try:
            mta_dfs = []
            for route in mta_routes:
                mta_params = {"key": MTA_KEY, "LineRef": f"MTA NYCT_{route}"}  # needs to be in the loop
                response = requests.get(mta_url, params=mta_params)
                delivery = response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]
                mta_vehicles = delivery.get('VehicleActivity', [])  # empty list if no active vehicles
                mta_rows = []
                for vehicle in mta_vehicles:
                    mvj = vehicle['MonitoredVehicleJourney']  # the path to data
                    if 'MonitoredCall' not in mvj:  # skip buses in layover
                        continue
                    mc = mvj['MonitoredCall']  # the path to data
                    if 'ExpectedArrivalTime' not in mc:  # skip vehicles with no prediction
                        continue
                    # parse both times so it can be subtracted to get dev
                    aimed = pd.to_datetime(mc['AimedArrivalTime'])
                    expected = pd.to_datetime(mc['ExpectedArrivalTime'])
                    row = {
                        'vehicle_id': mvj['VehicleRef'].replace('MTA NYCT_', ''),  # removing prefix
                        'lat': mvj['VehicleLocation']['Latitude'],
                        'lon': mvj['VehicleLocation']['Longitude'],
                        'direction': int(mvj['DirectionRef']),  # making the string "0"/"1" to int
                        'timestamp': pd.to_datetime(vehicle['RecordedAtTime']),
                        'route_id': mvj['LineRef'].replace('MTA NYCT_', ''),  # removing prefix
                        'call_distance': mc['Extensions']['Distances']['CallDistanceAlongRoute'],  # stop_sequence equivalent
                        'deviation': (expected - aimed).total_seconds() / 60  # positive = late, negative = early
                    }
                    mta_rows.append(row)
                df = pd.DataFrame(mta_rows)
                if df.empty:  # skip if no valid vehicles for this route
                    continue
                df['agency'] = 'MTA'
                df['timestamp'] = df['timestamp'].dt.tz_localize(None)  # making timezone to match WMATA/MBTA
                mta_dfs.append(df)
            if mta_dfs:
                mta_df = pd.concat(mta_dfs, ignore_index=True)
                mta_df.to_csv(mta_file, mode='a', header=not os.path.exists(mta_file), index=False)
        except Exception as e:
            print(f"MTA error: {e}")

        print(f"Poll complete: {pd.Timestamp.now().strftime('%H:%M:%S')}")
        time.sleep(60)  # wait 1 minute before next poll

    except KeyboardInterrupt:
        print("Polling stopped.")
        break

Poll complete: 23:04:33
Polling stopped.


### Note:

- Main data collection was done in a different file "api_script" as it auto saves and commits to two different repos
- The above loop will append anything ran to the existing data files (it won't breaking anything by running whats above)

## Cleaning

In [83]:
wmata = pd.read_csv("data/wmata.csv")
mbta = pd.read_csv("data/mbta.csv")
mta = pd.read_csv("data/mta.csv")

/var/folders/mn/ysmjqw4s37x8_rsj533gmd3r0000gn/T/ipykernel_7116/643784603.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  mbta = pd.read_csv("data/mbta.csv")


In [84]:
print(wmata.groupby('route_id').size().to_string())
print(mbta.groupby('route_id').size().to_string())
print(mta.groupby('route_id').size().to_string())

route_id
C23    23703
C35    24196
C53    95763
C61    48622
C81    29142
D20    48028
D24    26666
D40    69281
D60    54108
D80    96420
P14    18978
P16    24227
P20    18555
P40    42339
route_id
1       58530
8         123
19        200
21         27
23      63977
28      61538
36        202
37        310
38        360
39      53910
42      12879
50       6542
57      57424
64      21645
66      76724
70      48241
85       6115
87      20112
111     67509
201      7304
4050      532
route_id
B32        9642
B41      171006
B44       84577
B46      105249
B84        6403
BX12      39044
BX42      44974
M14A+     45902
M21       14939
M23+      39156
M50       16683
Q42        6764
Q67       24338
Q98       56701


#### WMATA

In [85]:
wmata

,vehicle_id,lat,lon,deviation,timestamp,trip_id,route_id,direction,trip_start_time,agency
0,5509,38.900446,-77.021919,1.0,2026-03-31 12:06:16,4927020,D40,1,2026-03-31 11:12:00,WMATA
1,5515,38.975230,-77.026732,9.0,2026-03-31 12:06:27,6977020,D40,0,2026-03-31 11:12:00,WMATA
2,5522,38.948387,-77.027067,17.0,2026-03-31 12:06:31,25982020,D40,1,2026-03-31 11:24:00,WMATA
3,5517,38.972468,-77.027016,-2.0,2026-03-31 12:06:23,4651020,D40,0,2026-03-31 11:24:00,WMATA
4,5507,38.939690,-77.024966,2.0,2026-03-31 12:06:33,28529020,D40,0,2026-03-31 11:36:00,WMATA
...,...,...,...,...,...,...,...,...,...,...
620023,3315,38.906613,-76.926932,0.0,2026-04-06 22:03:28,29678080,C23,1,2026-04-06 22:00:00,WMATA
620024,7017,39.004999,-76.907630,48.0,2026-04-06 22:03:21,30064080,P20,0,2026-04-06 20:30:00,WMATA
620025,7015,38.957943,-76.868802,57.0,2026-04-06 22:03:04,14090080,P20,0,2026-04-06 21:00:00,WMATA
620026,4775,38.952599,-76.878342,0.0,2026-04-06 22:03:31,7743080,P20,0,2026-04-06 22:00:00,WMATA


In [86]:
print(wmata.head())
print(wmata.dtypes)
print(wmata.isnull().sum())

   vehicle_id        lat        lon  deviation            timestamp   trip_id  \
0        5509  38.900446 -77.021919        1.0  2026-03-31 12:06:16   4927020   
1        5515  38.975230 -77.026732        9.0  2026-03-31 12:06:27   6977020   
2        5522  38.948387 -77.027067       17.0  2026-03-31 12:06:31  25982020   
3        5517  38.972468 -77.027016       -2.0  2026-03-31 12:06:23   4651020   
4        5507  38.939690 -77.024966        2.0  2026-03-31 12:06:33  28529020   

  route_id  direction      trip_start_time agency  
0      D40          1  2026-03-31 11:12:00  WMATA  
1      D40          0  2026-03-31 11:12:00  WMATA  
2      D40          1  2026-03-31 11:24:00  WMATA  
3      D40          0  2026-03-31 11:24:00  WMATA  
4      D40          0  2026-03-31 11:36:00  WMATA  
vehicle_id           int64
lat                float64
lon                float64
deviation          float64
timestamp           object
trip_id              int64
route_id            object
direction   

In [87]:
# fixing dtype
wmata['timestamp'] = pd.to_datetime(wmata['timestamp'])
wmata['trip_start_time'] = pd.to_datetime(wmata['trip_start_time'])
wmata['vehicle_id'] = wmata['vehicle_id'].astype(str)
wmata['trip_id'] = wmata['trip_id'].astype(str)

print(wmata.dtypes)

vehicle_id                 object
lat                       float64
lon                       float64
deviation                 float64
timestamp          datetime64[ns]
trip_id                    object
route_id                   object
direction                   int64
trip_start_time    datetime64[ns]
agency                     object
dtype: object


In [88]:
# Checking for any duplicates(same vehicle at exact same timestamp)
dupes = wmata.duplicated(subset=['vehicle_id', 'timestamp']).sum()
print(f"Duplicates: {dupes}")

Duplicates: 6764


In [91]:
wmata = wmata.drop_duplicates(subset=['vehicle_id', 'timestamp']).copy()
print(wmata.shape)

(613264, 13)


In [93]:
# Creating peak hour and day of the week vars
wmata['hour'] = wmata['timestamp'].dt.hour
wmata['day_of_week'] = wmata['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday
wmata['peak'] = wmata['hour'].apply(lambda x: 1 if (7 <= x <= 10) or (16 <= x <= 19) else 0) #

print(wmata[['timestamp', 'hour', 'day_of_week', 'peak']].head(10))
print(wmata['peak'].value_counts())

            timestamp  hour  day_of_week  peak
0 2026-03-31 12:06:16    12            1     0
1 2026-03-31 12:06:27    12            1     0
2 2026-03-31 12:06:31    12            1     0
3 2026-03-31 12:06:23    12            1     0
4 2026-03-31 12:06:33    12            1     0
5 2026-03-31 12:06:33    12            1     0
6 2026-03-31 12:06:14    12            1     0
7 2026-03-31 12:06:27    12            1     0
8 2026-03-31 12:06:28    12            1     0
9 2026-03-31 12:06:32    12            1     0
peak
0    337426
1    275838
Name: count, dtype: int64


In [94]:
# Downloading GTFS Static for bunching indicator
import zipfile
import io

gtfs_response = requests.get("https://api.wmata.com/gtfs/bus-gtfs-static.zip", headers=wmata_headers)
z = zipfile.ZipFile(io.BytesIO(gtfs_response.content))
print(z.namelist()) 

['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'feed_info.txt', 'routes.txt', 'shapes.txt', 'stops.txt', 'stop_times.txt', 'timepoints.txt', 'timepoint_times.txt', 'trips.txt']


In [96]:
# stop_times has scheduled arrivals but no route_id, trips.txt connects trip_id to route_id
stop_times = pd.read_csv(z.open('stop_times.txt'))
trips = pd.read_csv(z.open('trips.txt'))
print(stop_times.head())
print(stop_times.dtypes)
print(trips.head())
print(trips.dtypes)

   trip_id arrival_time departure_time  stop_id  stop_sequence  stop_headsign  \
0     1070      6:00:00        6:00:00     8436              2            NaN   
1     1070      6:01:22        6:01:22    21776              3            NaN   
2     1070      6:01:56        6:01:56     8328              4            NaN   
3     1070      6:02:38        6:02:38    21777              5            NaN   
4     1070      6:03:10        6:03:10    19149              6            NaN   

   pickup_type  drop_off_type  shape_dist_traveled  timepoint  
0            0              0                  NaN          1  
1            0              0                  NaN          0  
2            0              0                  NaN          0  
3            0              0                  NaN          0  
4            0              0                  NaN          0  
trip_id                  int64
arrival_time            object
departure_time          object
stop_id                  int64
stop_

In [97]:
# filtering to routes needed
# filter trips to only our routes
wmata_trips = trips[trips['route_id'].isin(wmata_routes)]
print(wmata_trips.shape)
print(wmata_trips['route_id'].unique())

(16767, 7)
['D20' 'C35' 'C61' 'D24' 'C81' 'P16' 'P20' 'P40' 'P14' 'D40' 'D60' 'D80'
 'C53' 'C23']


In [98]:
scheduled = stop_times.merge(wmata_trips[['trip_id', 'route_id', 'direction_id']], on='trip_id')
# keeping only first stop per trip since it gives the scheduled departure time for each trip
scheduled = scheduled[scheduled['stop_sequence'] == scheduled.groupby('trip_id')['stop_sequence'].transform('min')]
print(scheduled.shape)
print(scheduled.head())

(16767, 12)
     trip_id arrival_time departure_time  stop_id  stop_sequence  \
0       1070      6:00:00        6:00:00     8436              2   
31      1100      6:00:00        6:00:00     8436              2   
62      2020      5:00:00        5:00:00    11991              2   
92      2040      5:00:00        5:00:00    11991              2   
122     2060      5:00:00        5:00:00    11991              2   

     stop_headsign  pickup_type  drop_off_type  shape_dist_traveled  \
0              NaN            0              0                  NaN   
31             NaN            0              0                  NaN   
62             NaN            0              0                  NaN   
92             NaN            0              0                  NaN   
122            NaN            0              0                  NaN   

     timepoint route_id  direction_id  
0            1      C61             0  
31           1      C61             0  
62           1      C61         

In [99]:
#trips past midnight use times like 25:30:00 instead of 01:30:00 to stay on the same service day on GTFS
scheduled['departure_time'] = pd.to_timedelta(scheduled['departure_time']) # convert departure_time to timedelta since GTFS times can exceed 24:00:00
scheduled = scheduled.sort_values(['route_id', 'direction_id', 'departure_time'])
# calc scheduled headway as gap between consecutive trips on same route/direction (in minutes)
scheduled['scheduled_headway'] = scheduled.groupby(['route_id', 'direction_id'])['departure_time'].diff().dt.total_seconds() / 60

print(scheduled[['route_id', 'direction_id', 'departure_time', 'scheduled_headway']].head(20))

       route_id  direction_id  departure_time  scheduled_headway
331319      C23             0 0 days 06:00:00                NaN
331365      C23             0 0 days 06:00:00                0.0
331411      C23             0 0 days 06:00:00                0.0
331457      C23             0 0 days 06:00:00                0.0
331503      C23             0 0 days 06:00:00                0.0
473534      C23             0 0 days 06:00:00                0.0
473580      C23             0 0 days 06:00:00                0.0
616070      C23             0 0 days 06:00:00                0.0
350581      C23             0 0 days 06:30:00               30.0
350627      C23             0 0 days 06:30:00                0.0
449239      C23             0 0 days 06:30:00                0.0
657396      C23             0 0 days 06:30:00                0.0
657442      C23             0 0 days 06:30:00                0.0
657488      C23             0 0 days 06:30:00                0.0
657534      C23          

In [101]:
# multiple trips have the same departure time, shouldn't be the case
print(trips['service_id'].value_counts())
calendar = pd.read_csv(z.open('calendar.txt')) # which service IDs correspond to what day
print(calendar)

service_id
10    13428
9     13425
6     13421
1     13383
3     13379
4     10406
8     10302
2      9840
Name: count, dtype: int64
   service_id  monday  tuesday  wednesday  thursday  friday  saturday  sunday  \
0           1       0        0          0         0       0         0       0   
1           2       0        0          0         0       0         0       1   
2           3       0        0          0         0       0         0       0   
3           4       0        0          0         0       0         0       0   
4           6       0        0          0         0       1         0       0   
5           8       0        0          0         0       0         1       0   
6           9       1        1          0         1       0         0       0   
7          10       0        0          1         0       0         0       0   

   start_date  end_date  
0    20260315  20260613  
1    20260315  20260613  
2    20260315  20260613  
3    20260315  20260613  
4    20

In [102]:
# keeping only service IDs with at least one active day
active_serviced = calendar[calendar[['monday','tuesday','wednesday','thursday','friday','saturday','sunday']].sum(axis=1) > 0]['service_id']
print(active_serviced.values)

[ 2  6  8  9 10]


In [104]:
# filtering to only active service days
wmata_trips = trips[trips['route_id'].isin(wmata_routes) & trips['service_id'].isin(active_serviced)]
print(wmata_trips.shape)
print(wmata_trips['service_id'].value_counts())

(10432, 7)
service_id
6     2221
9     2221
10    2221
8     1927
2     1842
Name: count, dtype: int64


In [105]:
scheduled = stop_times.merge(wmata_trips[['trip_id', 'route_id', 'direction_id', 'service_id']], on='trip_id')
scheduled = scheduled[scheduled['stop_sequence'] == scheduled.groupby('trip_id')['stop_sequence'].transform('min')]

scheduled['departure_time'] = pd.to_timedelta(scheduled['departure_time'])

scheduled = scheduled.sort_values(['route_id', 'direction_id', 'service_id', 'departure_time'])
scheduled['scheduled_headway'] = scheduled.groupby(['route_id', 'direction_id', 'service_id'])['departure_time'].diff().dt.total_seconds() / 60

print(scheduled[['route_id', 'direction_id', 'service_id', 'departure_time', 'scheduled_headway']].head(20))

       route_id  direction_id  service_id  departure_time  scheduled_headway
382304      C23             0           2 0 days 06:00:00                NaN
278159      C23             0           2 0 days 06:30:00               30.0
362798      C23             0           2 0 days 07:00:00               30.0
187216      C23             0           2 0 days 07:30:00               30.0
74341       C23             0           2 0 days 08:00:00               30.0
357219      C23             0           2 0 days 08:30:00               30.0
279914      C23             0           2 0 days 09:00:00               30.0
2575        C23             0           2 0 days 09:30:00               30.0
145642      C23             0           2 0 days 10:00:00               30.0
104486      C23             0           2 0 days 10:30:00               30.0
184908      C23             0           2 0 days 11:00:00               30.0
401140      C23             0           2 0 days 11:30:00               30.0

In [106]:
print(wmata['day_of_week'].value_counts().sort_index())

day_of_week
0     95019
1     45810
2    103183
3    103097
4     99448
5     89685
6     77022
Name: count, dtype: int64


In [107]:
# mapping day of week to GTFS service_id based on calendar
day_to_service = {0: 9, 1: 9, 2: 10, 3: 9, 4: 6, 5: 8, 6: 2}
wmata['service_id'] = wmata['day_of_week'].map(day_to_service)
print(wmata['service_id'].value_counts())

service_id
9     243926
10    103183
6      99448
8      89685
2      77022
Name: count, dtype: int64


In [108]:
# converting trip_start_time to timedelta to match GTFS departure_time format
wmata['trip_start_timedelta'] = pd.to_timedelta(wmata['trip_start_time'].dt.strftime('%H:%M:%S'))
print(wmata[['trip_start_time', 'trip_start_timedelta']].head())

      trip_start_time trip_start_timedelta
0 2026-03-31 11:12:00      0 days 11:12:00
1 2026-03-31 11:12:00      0 days 11:12:00
2 2026-03-31 11:24:00      0 days 11:24:00
3 2026-03-31 11:24:00      0 days 11:24:00
4 2026-03-31 11:36:00      0 days 11:36:00


In [109]:
# adding scheduled headway to RT data by route, direction, service day, & trip start time
wmata = wmata.merge(
    scheduled[['route_id', 'direction_id', 'service_id', 'departure_time', 'scheduled_headway']],
    left_on=['route_id', 'direction', 'service_id', 'trip_start_timedelta'],
    right_on=['route_id', 'direction_id', 'service_id', 'departure_time'],
    how='left'
)
print(wmata.shape)
print(wmata['scheduled_headway'].isnull().sum())

(614675, 18)
21468


In [112]:
# which routes have the most nulls
print(wmata[wmata['scheduled_headway'].isnull()].groupby('route_id').size().sort_values(ascending=False))

print(wmata[wmata['scheduled_headway'].isnull()].groupby('route_id')['trip_start_time'].agg(['min', 'max', 'mean']))

route_id
C53    4484
D40    2857
D60    2842
D80    2339
D20    1652
P40    1632
C61    1094
C23    1022
C81     786
P16     606
P20     555
D24     541
P14     534
C35     524
dtype: int64
                         min                 max                          mean
route_id                                                                      
C23      2026-04-01 06:00:00 2026-04-07 00:00:00 2026-04-03 23:05:19.373776896
C35      2026-04-01 04:41:00 2026-04-07 00:00:00 2026-04-03 17:22:35.381679360
C53      2026-04-01 00:00:00 2026-04-07 00:00:00 2026-04-03 23:45:24.955396864
C61      2026-04-01 00:00:00 2026-04-07 00:00:00 2026-04-03 21:31:57.367458816
C81      2026-04-01 00:00:00 2026-04-07 00:00:00 2026-04-04 02:33:19.847328256
D20      2026-04-01 00:00:00 2026-04-07 00:00:00 2026-04-03 22:14:39.697336320
D24      2026-03-31 00:04:00 2026-04-07 00:00:00 2026-04-03 17:37:14.306839296
D40      2026-04-01 00:00:00 2026-04-07 00:00:00 2026-04-03 21:22:58.676933632
D60      2026-04-01 

In [113]:
# dropping unmatched rows these are mostly around midnight RT and GTFS date/times don't align
wmata = wmata.dropna(subset=['scheduled_headway'])
print(wmata.shape)

(593207, 18)


In [114]:
# bunching indicator: 1 if actual headway < 50% of scheduled headway
wmata['bunching'] = (wmata['deviation'] > wmata['scheduled_headway'] * 0.5).astype(int)
print(wmata['bunching'].value_counts())

bunching
0    473429
1    119778
Name: count, dtype: int64


In [115]:
wmata

,vehicle_id,lat,lon,deviation,timestamp,trip_id,route_id,direction,trip_start_time,agency,hour,day_of_week,peak,service_id,trip_start_timedelta,direction_id,departure_time,scheduled_headway,bunching
0,5509,38.900446,-77.021919,1.0,2026-03-31 12:06:16,4927020,D40,1,2026-03-31 11:12:00,WMATA,12,1,0,9,0 days 11:12:00,1.0,0 days 11:12:00,12.0,0
1,5515,38.975230,-77.026732,9.0,2026-03-31 12:06:27,6977020,D40,0,2026-03-31 11:12:00,WMATA,12,1,0,9,0 days 11:12:00,0.0,0 days 11:12:00,12.0,1
2,5522,38.948387,-77.027067,17.0,2026-03-31 12:06:31,25982020,D40,1,2026-03-31 11:24:00,WMATA,12,1,0,9,0 days 11:24:00,1.0,0 days 11:24:00,12.0,1
3,5517,38.972468,-77.027016,-2.0,2026-03-31 12:06:23,4651020,D40,0,2026-03-31 11:24:00,WMATA,12,1,0,9,0 days 11:24:00,0.0,0 days 11:24:00,12.0,0
4,5507,38.939690,-77.024966,2.0,2026-03-31 12:06:33,28529020,D40,0,2026-03-31 11:36:00,WMATA,12,1,0,9,0 days 11:36:00,0.0,0 days 11:36:00,12.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
614670,3315,38.906613,-76.926932,0.0,2026-04-06 22:03:28,29678080,C23,1,2026-04-06 22:00:00,WMATA,22,0,0,9,0 days 22:00:00,1.0,0 days 22:00:00,30.0,0
614671,7017,39.004999,-76.907630,48.0,2026-04-06 22:03:21,30064080,P20,0,2026-04-06 20:30:00,WMATA,22,0,0,9,0 days 20:30:00,0.0,0 days 20:30:00,30.0,1
614672,7015,38.957943,-76.868802,57.0,2026-04-06 22:03:04,14090080,P20,0,2026-04-06 21:00:00,WMATA,22,0,0,9,0 days 21:00:00,0.0,0 days 21:00:00,30.0,1
614673,4775,38.952599,-76.878342,0.0,2026-04-06 22:03:31,7743080,P20,0,2026-04-06 22:00:00,WMATA,22,0,0,9,0 days 22:00:00,0.0,0 days 22:00:00,60.0,0


In [117]:
wmata.to_csv("data/wmata_v1.csv", index=False)
print("saved")

saved


In [118]:
wmata = wmata.drop(columns=['trip_start_timedelta', 'direction_id', 'departure_time'])
wmata

,vehicle_id,lat,lon,deviation,timestamp,trip_id,route_id,direction,trip_start_time,agency,hour,day_of_week,peak,service_id,scheduled_headway,bunching
0,5509,38.900446,-77.021919,1.0,2026-03-31 12:06:16,4927020,D40,1,2026-03-31 11:12:00,WMATA,12,1,0,9,12.0,0
1,5515,38.975230,-77.026732,9.0,2026-03-31 12:06:27,6977020,D40,0,2026-03-31 11:12:00,WMATA,12,1,0,9,12.0,1
2,5522,38.948387,-77.027067,17.0,2026-03-31 12:06:31,25982020,D40,1,2026-03-31 11:24:00,WMATA,12,1,0,9,12.0,1
3,5517,38.972468,-77.027016,-2.0,2026-03-31 12:06:23,4651020,D40,0,2026-03-31 11:24:00,WMATA,12,1,0,9,12.0,0
4,5507,38.939690,-77.024966,2.0,2026-03-31 12:06:33,28529020,D40,0,2026-03-31 11:36:00,WMATA,12,1,0,9,12.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
614670,3315,38.906613,-76.926932,0.0,2026-04-06 22:03:28,29678080,C23,1,2026-04-06 22:00:00,WMATA,22,0,0,9,30.0,0
614671,7017,39.004999,-76.907630,48.0,2026-04-06 22:03:21,30064080,P20,0,2026-04-06 20:30:00,WMATA,22,0,0,9,30.0,1
614672,7015,38.957943,-76.868802,57.0,2026-04-06 22:03:04,14090080,P20,0,2026-04-06 21:00:00,WMATA,22,0,0,9,30.0,1
614673,4775,38.952599,-76.878342,0.0,2026-04-06 22:03:31,7743080,P20,0,2026-04-06 22:00:00,WMATA,22,0,0,9,60.0,0


In [120]:
wmata.to_csv("data/wmata_clean.csv", index=False)
print("saved")

saved


#### MBTA

In [121]:
mbta

,vehicle_id,lat,lon,direction,timestamp,stop_sequence,trip_id,route_id,agency
0,y3251,42.364823,-71.102914,0,2026-03-31 12:06:40,18.0,73604104,1,MBTA
1,y1903,42.331403,-71.076767,1,2026-03-31 12:06:47,20.0,73604114,1,MBTA
2,y1875,42.358715,-71.093350,0,2026-03-31 12:06:38,15.0,73604103,1,MBTA
3,y1854,42.347308,-71.088111,1,2026-03-31 12:06:45,12.0,73604080,1,MBTA
4,y1849,42.332378,-71.079953,1,2026-03-31 12:06:45,21.0,73604086,1,MBTA
...,...,...,...,...,...,...,...,...,...
564199,y3158,42.378492,-71.092850,1,2026-04-06 22:03:45,31.0,75523316,87,MBTA
564200,y3154,42.381398,-71.099678,0,2026-04-06 22:03:43,7.0,75523472,87,MBTA
564201,y2105,42.363260,-71.132340,1,2026-04-06 22:03:40,39.0,75522396,70,MBTA
564202,y2063,42.364558,-71.175511,0,2026-04-06 22:03:41,22.0,75522317,70,MBTA


In [122]:
print(mbta.head())
print(mbta.dtypes)
print(mbta.isnull().sum())

  vehicle_id        lat        lon  direction            timestamp  \
0      y3251  42.364823 -71.102914          0  2026-03-31 12:06:40   
1      y1903  42.331403 -71.076767          1  2026-03-31 12:06:47   
2      y1875  42.358715 -71.093350          0  2026-03-31 12:06:38   
3      y1854  42.347308 -71.088111          1  2026-03-31 12:06:45   
4      y1849  42.332378 -71.079953          1  2026-03-31 12:06:45   

   stop_sequence   trip_id  route_id agency  
0           18.0  73604104         1   MBTA  
1           20.0  73604114         1   MBTA  
2           15.0  73604103         1   MBTA  
3           12.0  73604080         1   MBTA  
4           21.0  73604086         1   MBTA  
vehicle_id        object
lat              float64
lon              float64
direction          int64
timestamp         object
stop_sequence    float64
trip_id           object
route_id           int64
agency            object
dtype: object
vehicle_id        0
lat               0
lon               0
dire

In [125]:
# filtering to routes in mbta_routes
mbta = mbta[mbta['route_id'].isin([int(r) for r in mbta_routes if r.isdigit()] + mbta_routes)].copy()
print(mbta['route_id'].unique())

['1' '23' '28' '39' '57' '66' '111' '64' '201' '50' '42' '87' '70' '85']


In [126]:
mbta['timestamp'] = pd.to_datetime(mbta['timestamp'])
mbta['route_id'] = mbta['route_id'].astype(str)
mbta['trip_id'] = mbta['trip_id'].astype(str)

print(mbta.dtypes)

vehicle_id               object
lat                     float64
lon                     float64
direction                 int64
timestamp        datetime64[ns]
stop_sequence           float64
trip_id                  object
route_id                 object
agency                   object
dtype: object


In [127]:
print(mbta[mbta['stop_sequence'].isnull()][['route_id', 'vehicle_id', 'timestamp']].head(10))

       route_id vehicle_id           timestamp
123156       87      y2029 2026-04-01 19:15:09
256831       66      y1849 2026-04-03 08:05:40
256943       66      y1849 2026-04-03 08:06:57
367801       70      y2015 2026-04-04 13:37:51
367972       70      y2015 2026-04-04 13:40:06
369098       70      y2099 2026-04-04 13:54:29
369184       70      y2099 2026-04-04 13:55:18
369360       70      y2099 2026-04-04 13:57:47
369541       70      y2021 2026-04-04 14:00:03
369625       70      y2021 2026-04-04 14:01:08


In [128]:
# dropping rows with missing stop_sequence 
mbta = mbta.dropna(subset=['stop_sequence'])
mbta['stop_sequence'] = mbta['stop_sequence'].astype(int)
print(mbta.shape)

(562366, 9)


In [130]:
# checking for duplicates
dupes = mbta.duplicated(subset=['vehicle_id', 'timestamp']).sum()
print(f"Duplicates: {dupes}")

mbta = mbta.drop_duplicates(subset=['vehicle_id', 'timestamp']).copy()
print(mbta.shape)

Duplicates: 6061
(556305, 9)


In [131]:
# peak hour and day vars
mbta['hour'] = mbta['timestamp'].dt.hour  
mbta['day_of_week'] = mbta['timestamp'].dt.dayofweek  
mbta['peak'] = mbta['hour'].apply(lambda x: 1 if (7 <= x <= 10) or (16 <= x <= 19) else 0) # 1 if peak, 0 otherwise

print(mbta['peak'].value_counts())

peak
0    302571
1    253734
Name: count, dtype: int64


In [132]:
#GTFS Static
mbta_gtfs = requests.get("https://cdn.mbta.com/MBTA_GTFS.zip", headers=mbta_headers)
z_mbta = zipfile.ZipFile(io.BytesIO(mbta_gtfs.content))
print(z_mbta.namelist())

['agency.txt', 'areas.txt', 'calendar.txt', 'calendar_attributes.txt', 'calendar_dates.txt', 'checkpoints.txt', 'directions.txt', 'facilities.txt', 'facilities_properties.txt', 'facilities_properties_definitions.txt', 'fare_leg_join_rules.txt', 'fare_leg_rules.txt', 'fare_media.txt', 'fare_products.txt', 'fare_transfer_rules.txt', 'feed_info.txt', 'levels.txt', 'lines.txt', 'linked_datasets.txt', 'multi_route_trips.txt', 'pathways.txt', 'route_patterns.txt', 'routes.txt', 'shapes.txt', 'stop_areas.txt', 'stop_times.txt', 'stops.txt', 'timeframes.txt', 'transfers.txt', 'trips.txt', 'trips_properties.txt', 'trips_properties_definitions.txt']


In [133]:
mbta_stop_times = pd.read_csv(z_mbta.open('stop_times.txt'))
mbta_trips = pd.read_csv(z_mbta.open('trips.txt'))
print(mbta_trips.head())
print(mbta_trips.dtypes)
print(mbta_stop_times.head())
print(mbta_stop_times.dtypes)

/var/folders/mn/ysmjqw4s37x8_rsj533gmd3r0000gn/T/ipykernel_7116/179596945.py:1: DtypeWarning: Columns (0,3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  mbta_stop_times = pd.read_csv(z_mbta.open('stop_times.txt'))


  route_id                     service_id   trip_id trip_headsign  \
0        1  BUS12026-hbc16016-Saturday-02  73390421       Harvard   
1        1  BUS12026-hbc16016-Saturday-02  73390422       Harvard   
2        1  BUS12026-hbc16016-Saturday-02  73390423       Harvard   
3        1  BUS12026-hbc16016-Saturday-02  73390424       Harvard   
4        1  BUS12026-hbc16016-Saturday-02  73390425       Harvard   

  trip_short_name  direction_id block_id shape_id  wheelchair_accessible  \
0             NaN             0    C01-4   010154                      1   
1             NaN             0   C01-14   010154                      1   
2             NaN             0   C01-14   010154                      1   
3             NaN             0    C01-3   010154                      1   
4             NaN             0    C01-2   010154                      1   

   trip_route_type route_pattern_id  bikes_allowed  
0              NaN            1-_-0              1  
1              NaN    

/var/folders/mn/ysmjqw4s37x8_rsj533gmd3r0000gn/T/ipykernel_7116/179596945.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  mbta_trips = pd.read_csv(z_mbta.open('trips.txt'))


In [135]:
# Checking service ID structure
mbta_calendar = pd.read_csv(z_mbta.open('calendar.txt'))
print(mbta_calendar.head(10))

                      service_id  monday  tuesday  wednesday  thursday  \
0  BUS12026-hbb16016-Saturday-02       0        0          0         0   
1   BUS12026-hbb16fr1-Weekday-02       0        0          0         0   
2   BUS12026-hbb16wk1-Weekday-02       1        1          1         1   
3  BUS12026-hbc16016-Saturday-02       0        0          0         0   
4   BUS12026-hbc16fr1-Weekday-02       0        0          0         0   
5   BUS12026-hbc16wk1-Weekday-02       1        1          1         1   
6  BUS12026-hbg16016-Saturday-02       0        0          0         0   
7   BUS12026-hbg16fr1-Weekday-02       0        0          0         0   
8   BUS12026-hbg16wk1-Weekday-02       1        1          1         1   
9  BUS12026-hbl16016-Saturday-02       0        0          0         0   

   friday  saturday  sunday  start_date  end_date  
0       0         1       0    20260404  20260404  
1       1         0       0    20260403  20260403  
2       0         0       0  

## Resources

- Standardizing time zones:

https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.tz_localize.html

- If continue and break statements:

https://www.educative.io/answers/what-are-break-continue-and-pass-statements-in-python?utm_campaign=Pmax_feb25&utm_source=google&utm_medium=ppc&utm_content=&utm_term=&eid=5082902844932096&utm_term=&utm_campaign=%5BMar+25%5D+Pmax.+-+Coding+Interview+Prep&utm_source=adwords&utm_medium=ppc&hsa_acc=5451446008&hsa_cam=22344713166&hsa_grp=&hsa_ad=&hsa_src=x&hsa_tgt=&hsa_kw=&hsa_mt=&hsa_net=adwords&hsa_ver=3&gad_source=1&gad_campaignid=22354833079&gbraid=0AAAAADfWLuSny8RMeOlF5ThZ04eXKwptZ&gclid=CjwKCAjwvqjOBhAGEiwAngeQnVWlwZMFRhHCXD6ekkGarMTrSQa1YzFdylNc3-Rritp4jo-j1r1iWRoCCMoQAvD_BwE

- Timing Loops

https://www.digitalocean.com/community/tutorials/python-time-sleep

- Try Except

https://www.w3schools.com/python/python_try_except.asp
https://www.reddit.com/r/learnpython/comments/1blfl9e/explain_try_except_structure_in_practical_examples/
https://stackoverflow.com/questions/62062226/how-to-work-with-try-and-except-in-python
https://docs.python.org/3/tutorial/errors.html   
 https://stackoverflow.com/questions/21120947/catching-keyboardinterrupt-in-python-during-program-shutdown

- Auto Commit

https://docs.github.com/en/authentication/connecting-to-github-with-ssh
 https://docs.github.com/en/authentication/connecting-to-github-with-ssh/generating-a-new-ssh-key-and-adding-it-to-the-ssh-agent
 https://discourse.jupyter.org/t/how-to-run-a-notebook-using-command-line/3475
 https://stackoverflow.com/questions/5620525/git-pushing-to-two-repos-in-one-command
 https://stackoverflow.com/questions/4255865/git-push-to-multiple-repositories-simultaneously
 

- Peak Hour & Day

https://www.reddit.com/r/learnpython/comments/1g0ktqk/can_someone_explain_lambda_to_a_beginner/
https://stackoverflow.com/questions/42977395/pandas-dt-hour-formatting
https://docs.dask.org/en/latest/generated/dask.dataframe.Series.dt.dayofweek.html 
https://stackoverflow.com/questions/69973084/is-there-a-difference-between-pd-series-dt-weekday-vs-pd-series-dt-dayofweek 

- GTFS Reference

https://gtfs.org/documentation/realtime/reference/